# 🏦 VITBAIH Bank Marketing AI — End-to-End Notebook

This notebook consolidates the entire pipeline: Dataset loading, EDA, Plots, Business insights, Preprocessing, Model Training, Evaluation, and Feature Importance.

## 1. Environment Setup & Imports

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score, roc_curve
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")
%matplotlib inline

# Style config
DARK_BG = "#1a1a2e"
ACCENT = "#16213e"
PALETTE = {"yes": "#2ecc71", "no": "#e74c3c"}
plt.rcParams['figure.facecolor'] = DARK_BG
plt.rcParams['axes.facecolor'] = ACCENT
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'


## 2. Dataset Loading

In [ ]:
ROOT = Path("..").resolve()
DATA_PATH = ROOT / "data" / "bank-additional-full.csv"

df = pd.read_csv(DATA_PATH, sep=";")
print(f"Dataset Shape: {df.shape}")
df.head()


## 3. Exploratory Data Analysis (EDA)

In [ ]:
print("Missing values:", df.isnull().sum().sum())
print("Duplicates:", df.duplicated().sum())
print("\nTarget distribution:\n", df['y'].value_counts())
print(f"\nImbalance ratio: {df['y'].value_counts()['no'] / df['y'].value_counts()['yes']:.2f}:1")
df.describe().T


## 4. Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Target Distribution
counts = df['y'].value_counts()
axes[0].bar(counts.index, counts.values, color=[PALETTE[k] for k in counts.index], edgecolor="white")
axes[0].set_title("Target Distribution")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 500, f"{v:,}\n({v/len(df)*100:.1f}%)", ha="center", color="white")

# Age Distribution
for label, grp in df.groupby("y"):
    axes[1].hist(grp["age"], bins=30, alpha=0.7, label=label, color=PALETTE[label])
axes[1].set_title("Age Distribution by Subscription")
axes[1].legend()

plt.tight_layout()
plt.show()


## 5. Business Insights

In [ ]:
print("BUSINESS INSIGHTS\n" + "="*50)

# Job subscription rates
job_rate = (df.groupby("job")["y"].apply(lambda x: (x == "yes").mean() * 100)).sort_values(ascending=False)
print(f"[1] Highest subscription rate by job:\n{job_rate.round(2)}")

# Duration impact
dur_yes = df[df["y"] == "yes"]["duration"].median()
dur_no  = df[df["y"] == "no"]["duration"].median()
print(f"\n[2] Median duration — Yes: {dur_yes:.0f}s | No: {dur_no:.0f}s")
print(f"    Longer calls correlate with subscription (ratio: {dur_yes/dur_no:.2f}x)")


## 6. Preprocessing

In [ ]:
CAT_FEATURES = [
    "job", "marital", "education", "default",
    "housing", "loan", "contact", "month",
    "day_of_week", "poutcome",
]

df_encoded = df.copy()
df_encoded["y"] = df_encoded["y"].map({"yes": 1, "no": 0})
X = df_encoded.drop(columns=["y"])
y = df_encoded["y"]

for col in CAT_FEATURES:
    X[col] = X[col].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")


## 7. Model Training

In [ ]:
# Logistic Regression Baseline
print("Training Logistic Regression Baseline...")
ct = ColumnTransformer([
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CAT_FEATURES)
], remainder="passthrough")

lr_model = Pipeline([
    ("ct", ct),
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000, class_weight="balanced", solver="lbfgs", random_state=42))
])
lr_model.fit(X_train, y_train)

# CatBoost Main Model
print("Training CatBoost...")
cat_idx = [X_train.columns.tolist().index(c) for c in CAT_FEATURES]
train_pool = Pool(X_train, y_train, cat_features=cat_idx)
eval_pool  = Pool(X_test,  y_test,  cat_features=cat_idx)

cat_model = CatBoostClassifier(
    iterations=500, learning_rate=0.05, depth=6,
    eval_metric="AUC", auto_class_weights="Balanced",
    random_seed=42, early_stopping_rounds=30, verbose=100
)
cat_model.fit(train_pool, eval_set=eval_pool, use_best_model=True)
print("Training Complete.")


## 8. Evaluation & Feature Importance

In [ ]:
# Evaluate LR
lr_pred = lr_model.predict(X_test)
lr_prob = lr_model.predict_proba(X_test)[:, 1]

# Evaluate CatBoost
cat_pred = cat_model.predict(eval_pool).flatten()
cat_prob = cat_model.predict_proba(eval_pool)[:, 1]

def print_metrics(name, y_true, y_pred, y_prob):
    print(f"--- {name} ---")
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall   : {recall_score(y_true, y_pred):.4f}")
    print(f"F1 Score : {f1_score(y_true, y_pred):.4f}")
    print(f"ROC-AUC  : {roc_auc_score(y_true, y_prob):.4f}\n")

print_metrics("Logistic Regression", y_test, lr_pred, lr_prob)
print_metrics("CatBoost", y_test, cat_pred, cat_prob)

# Feature Importance
importances = cat_model.get_feature_importance()
fi = pd.Series(importances, index=X_train.columns).sort_values(ascending=True).tail(15)

plt.figure(figsize=(10, 6))
fi.plot(kind="barh", color="#3498db")
plt.title("Top 15 Feature Importances — CatBoost")
plt.xlabel("Importance Score")
plt.show()
